DeepEval Agentic Metrics Learning

Agentic metrics are used to evaluate AI applications that behave like agents.

An agent usually does:
user request
→ plans or decides steps
→ selects tools
→ calls tools
→ uses tool results
→ gives final answer

Agentic metrics check:
- Did the agent complete the task?
- Did the agent choose the correct tool?
- Did the agent pass correct arguments?
- Did the agent follow the plan?
- Did the agent avoid unnecessary steps?

This notebook is created separately for learning Agentic Metrics.

In [17]:
!pip install deepeval groq -q

In [18]:
import os
from google.colab import userdata
from groq import Groq

from deepeval.models import DeepEvalBaseLLM

In [19]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


class GroqDeepEvalModel(DeepEvalBaseLLM):

    def __init__(self, model_name="openai/gpt-oss-20b"):
        self.model_name = model_name
        self.client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def load_model(self):
        return self.client

    def generate(self, prompt: str, **kwargs) -> str:
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "system",
                    "content": "Return only valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        return response.choices[0].message.content

    async def a_generate(self, prompt: str, **kwargs) -> str:
        return self.generate(prompt, **kwargs)

    def get_model_name(self):
        return self.model_name


groq_model = GroqDeepEvalModel()

print("Groq evaluator connected.")

Groq evaluator connected.


In [39]:
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.evaluate import ErrorConfig, AsyncConfig
from deepeval.tracing import observe, update_current_trace
from deepeval.metrics import TaskCompletionMetric, StepEfficiencyMetric, ToolCorrectnessMetric, ArgumentCorrectnessMetric, PlanQualityMetric, PlanAdherenceMetric

TaskCompletionMetric

TaskCompletionMetric checks whether an AI agent successfully completed the user’s task.

It focuses on the final outcome of the agent’s work.

If the agent’s response fully satisfies the user request, it passes.

If the agent gives only partial help, unclear help, or does not complete the requested task, it fails.

In [21]:
task_completion_metric = TaskCompletionMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Task Completion metric created.")

Task Completion metric created.


In [22]:
agent_goldens = [
    Golden(input="Help the customer track their order."),
    Golden(input="Help the customer understand the refund policy."),
    Golden(input="Help the customer change their delivery address before shipment.")
]

agent_dataset = EvaluationDataset(goldens=agent_goldens)

print("Agent task dataset created.")

Agent task dataset created.


In [23]:
def track_order_tool():
    return "Order can be tracked using the tracking link sent to the registered email or mobile number."


def refund_policy_tool():
    return "Refund is allowed within 15 days if the item is unused and in original condition."


def change_address_tool():
    return "Delivery address can be changed before shipment by contacting customer support."

In [24]:
@observe()
def customer_support_agent(user_task):
    if "track" in user_task.lower():
        answer = track_order_tool()

    elif "refund" in user_task.lower():
        answer = refund_policy_tool()

    elif "delivery address" in user_task.lower() or "address" in user_task.lower():
        answer = change_address_tool()

    else:
        answer = "I could not identify the correct support action."

    update_current_trace(
        input=user_task,
        output=answer
    )

    return answer

In [25]:
for golden in agent_dataset.evals_iterator(
    metrics=[task_completion_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Agentic Task Completion evaluation completed.")

Output()

Task: Help the customer track their order.

Agent answer: Order can be tracked using the tracking link sent to the registered email or mobile number.

--------------------------------------------------------------------------------

Task: Help the customer understand the refund policy.

...

--------------------------------------------------------------------------------

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_3                                                                                                 │
│  ├──   Input:            Help the customer change their delivery address before shipment.                       │
│  │     Actual Output:    Delivery address can be changed before shipment by contacting customer support.        │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric          ┃ Score ┃ Threshold ┃ Reason                                                     │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Task Completion │ 0.35  │ 0.60      │ The response only informs the customer that the address    │
│              │                 │       │           │ can be changed by contacting support, but it does not      │
│              │                 │       │           │ provide any direct assistance, steps, or options to        │
│              │                 │       │           │ actually change the address within the system. Therefore   │
│              │                 │       │           │ it partially addresses the task but falls short of fully   │
│              │                 │       │           │ helping the customer.                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                    ┃ Average Score          ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Task Completion           │ 0.62                   │ 66.67% | passed=2 | failed=1                  │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=658390;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.93s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 66.67% | Passed: 2 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Agentic Task Completion evaluation completed.


StepEfficiencyMetric

StepEfficiencyMetric checks whether an AI agent completed the task using useful and necessary steps.

It focuses on the agent’s execution path.

If the agent uses direct and relevant steps, it passes.

If the agent uses unnecessary, repeated, or unrelated steps, it fails.

In [26]:
step_efficiency_metric = StepEfficiencyMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Step Efficiency metric created.")

Step Efficiency metric created.


In [27]:
for golden in agent_dataset.evals_iterator(
    metrics=[step_efficiency_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Agentic Step Efficiency evaluation completed.")

Output()

Task: Help the customer track their order.

Agent answer: Order can be tracked using the tracking link sent to the registered email or mobile number.

--------------------------------------------------------------------------------

Task: Help the customer understand the refund policy.

...

--------------------------------------------------------------------------------

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                    ┃ Average Score         ┃ Pass Rate                                      ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Step Efficiency           │ 1.00                  │ 100.00% | passed=3 | failed=0                  │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=338782;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 36.59s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Agentic Step Efficiency evaluation completed.


ToolCorrectnessMetric

ToolCorrectnessMetric checks whether an AI agent selected the correct tool for the given task.

It compares the tools actually used by the agent with the tools expected for that task.

If the agent calls the correct tool, it passes.

If the agent calls the wrong tool, misses a required tool, or uses an unnecessary tool, it fails.

In [28]:
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval import evaluate

In [29]:
tool_correctness_metric = ToolCorrectnessMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)
print("Tool Correctness metric created.")

Tool Correctness metric created.


In [30]:
tool_test_cases = [
    LLMTestCase(
        input="Help the customer track their order.",
        actual_output=track_order_tool(),
        tools_called=[ToolCall(name="track_order_tool")],
        expected_tools=[ToolCall(name="track_order_tool")]
    ),
    LLMTestCase(
        input="Help the customer understand the refund policy.",
        actual_output=refund_policy_tool(),
        tools_called=[ToolCall(name="refund_policy_tool")],
        expected_tools=[ToolCall(name="refund_policy_tool")]
    ),
    LLMTestCase(
        input="Help the customer change their delivery address before shipment.",
        actual_output=change_address_tool(),
        tools_called=[ToolCall(name="change_address_tool")],
        expected_tools=[ToolCall(name="change_address_tool")]
    )
]

evaluate(
    test_cases=tool_test_cases,
    metrics=[tool_correctness_metric]
)

print("Tool Correctness evaluation completed.")

✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Tool Correctness           │ 1.00                  │ 100.00% | passed=3 | failed=0                 │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=970323;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 0.15s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Tool Correctness evaluation completed.


ArgumentCorrectnessMetric

ArgumentCorrectnessMetric checks whether an AI agent passed the correct arguments or input values into the selected tool.

It is used after checking tool correctness.

If the agent selects the correct tool but passes wrong, missing, or incomplete arguments, this metric can fail.

Example:
If the task is “Track order ORD123”, the agent should call the tracking tool with order_id = "ORD123".

If the agent calls the tracking tool without the order ID, the tool choice is correct, but the argument is wrong.

In [32]:
argument_correctness_metric = ArgumentCorrectnessMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Argument Correctness metric created.")

Argument Correctness metric created.


In [34]:
argument_test_cases = [
    LLMTestCase(
        input="Track order ORD123.",
        actual_output="Order ORD123 can be tracked using the tracking link.",
        tools_called=[
            ToolCall(
                name="track_order_tool",
                input_parameters={"order_id": "ORD123"}
            )
        ]
    ),
    LLMTestCase(
        input="Check refund policy for order ORD456.",
        actual_output="Order ORD456 is eligible for refund if it is within 15 days and unused.",
        tools_called=[
            ToolCall(
                name="refund_policy_tool",
                input_parameters={"order_id": "ORD456"}
            )
        ]
    ),
    LLMTestCase(
        input="Change delivery address for order ORD789.",
        actual_output="Delivery address for order ORD789 can be changed before shipment.",
        tools_called=[
            ToolCall(
                name="change_address_tool",
                input_parameters={"order_id": "ORD789"}
            )
        ]
    )
]

evaluate(
    test_cases=argument_test_cases,
    metrics=[argument_correctness_metric]
)

print("Argument Correctness evaluation completed.")

✨ You're running DeepEval's latest Argument Correctness Metric! (using openai/gpt-oss-20b, strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:            Change delivery address for order ORD789.                                              │
│  │     Actual Output:    Delivery address for order ORD789 can be changed before shipment.                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Argument Correctness │ 0.00  │ 0.60      │ The score is 0.00 because the input only provides     │
│              │                      │       │           │ the order ID but does not include the new delivery    │
│              │                      │       │           │ address required to change the address.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                          ┃ Average Score        ┃ Pass Rate                                 ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Argument Correctness            │ 0.67                 │ 66.67% | passed=2 | failed=1              │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=98596;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.38s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 66.67% | Passed: 2 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Argument Correctness evaluation completed.


PlanQualityMetric

PlanQualityMetric checks whether an AI agent created a good plan before doing the task.

It focuses on the quality of the planned steps.

A good plan should be clear, logical, complete, and useful for completing the user’s task.

If the plan is missing important steps, has unnecessary steps, or is not useful for the task, this metric can fail.

In [36]:
plan_quality_metric = PlanQualityMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Plan Quality metric created.")

Plan Quality metric created.


In [37]:
@observe()
def planned_customer_support_agent(user_task):
    plan = [
        "Identify the customer support request.",
        "Select the correct support tool.",
        "Use the tool result to answer the customer."
    ]

    if "track" in user_task.lower():
        answer = track_order_tool()

    elif "refund" in user_task.lower():
        answer = refund_policy_tool()

    elif "delivery address" in user_task.lower() or "address" in user_task.lower():
        answer = change_address_tool()

    else:
        answer = "I could not identify the correct support action."

    update_current_trace(
        input=user_task,
        output=answer,
        metadata={
            "plan": plan
        }
    )

    return answer

In [38]:
for golden in agent_dataset.evals_iterator(
    metrics=[plan_quality_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = planned_customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Plan Quality evaluation completed.")

Output()

Task: Help the customer track their order.

Agent answer: Order can be tracked using the tracking link sent to the registered email or mobile number.

--------------------------------------------------------------------------------

Task: Help the customer understand the refund policy.

...

--------------------------------------------------------------------------------

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                ┃ Average Score           ┃ Pass Rate                                       ┃ Total      │
│ ━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━ │
│  Plan Quality          │ 1.00                    │ 100.00% | passed=3 | failed=0                   │ 3          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=677934;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.66s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Plan Quality evaluation completed.


PlanAdherenceMetric

PlanAdherenceMetric checks whether an AI agent followed the plan it created.

It compares the planned steps with the actual steps taken by the agent.

If the agent follows the planned steps properly, it passes.

If the agent skips planned steps, does different steps, or goes away from the plan, this metric can fail.

In [40]:
plan_adherence_metric = PlanAdherenceMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Plan Adherence metric created.")

Plan Adherence metric created.


In [41]:
for golden in agent_dataset.evals_iterator(
    metrics=[plan_adherence_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = planned_customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Plan Adherence evaluation completed.")

Output()

Task: Help the customer track their order.

Agent answer: Order can be tracked using the tracking link sent to the registered email or mobile number.

--------------------------------------------------------------------------------

Task: Help the customer understand the refund policy.

...

--------------------------------------------------------------------------------

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                   ┃ Average Score          ┃ Pass Rate                                      ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Plan Adherence           │ 1.00                   │ 100.00% | passed=3 | failed=0                  │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=249628;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.81s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Plan Adherence evaluation completed.
